# Fish Detection and Counting - Example Notebook

This notebook demonstrates how to use the fish migration monitoring system.

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Add src directory to path
sys.path.append('../src')

from utils.config_loader import load_config
from detection.fish_detector import FishDetector
from tracking.fish_tracker import FishTracker
from counting.fish_counter import FishCounter
from utils.video_processor import VideoProcessor

## 2. Load Configuration

In [ ]:
# Load configuration
config = load_config('../config/config.yaml')
print("Configuration loaded successfully")

## 3. Initialize Components

In [ ]:
# Initialize detector, tracker, and counter
detector = FishDetector(config)
tracker = FishTracker(config)
counter = FishCounter(config)

print("Components initialized")

## 4. Process Video

In [ ]:
# Set input and output paths
input_video = '../data/raw/sample_video.mp4'  # Change to your video path
output_dir = '../results/'

# Check if video exists
if not Path(input_video).exists():
    print(f"Video file not found: {input_video}")
    print("Please add a video file to the data/raw/ directory")
else:
    # Create video processor
    processor = VideoProcessor(
        input_path=input_video,
        output_path=output_dir,
        config=config,
        display=False  # Set to True to see processing in real-time
    )
    
    # Process the video
    results = processor.process(detector, tracker, counter)
    
    # Display results
    print("\nProcessing Complete!")
    print(f"Total fish detected: {results['total_fish']}")
    print(f"Fish swimming {results['upstream_label']}: {results['upstream_count']}")
    print(f"Fish swimming {results['downstream_label']}: {results['downstream_count']}")

## 5. Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Create visualization of results
if 'results' in locals():
    # Bar chart of counts
    labels = [results['upstream_label'].title(), results['downstream_label'].title()]
    counts = [results['upstream_count'], results['downstream_count']]
    
    plt.figure(figsize=(10, 6))
    plt.bar(labels, counts, color=['blue', 'red'])
    plt.title('Fish Migration Counts')
    plt.ylabel('Number of Fish')
    plt.xlabel('Direction')
    
    # Add count labels on bars
    for i, count in enumerate(counts):
        plt.text(i, count + 0.5, str(count), ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
else:
    print("No results to visualize. Process a video first.")

## 6. Analyze Statistics

In [ ]:
# Load statistics from CSV if available
import glob

csv_files = glob.glob('../results/statistics_*.csv')

if csv_files:
    # Load the most recent statistics file
    latest_csv = max(csv_files, key=Path.getctime)
    df = pd.read_csv(latest_csv)
    
    print("\nStatistics from CSV:")
    print(df.to_string(index=False))
else:
    print("No statistics files found in results directory")

## 7. Next Steps

To improve accuracy:
1. **Train a custom model**: Collect and annotate fish data, then train a YOLO model
2. **Tune parameters**: Adjust detection thresholds and tracking parameters in config.yaml
3. **Validate results**: Compare automated counts with manual counts
4. **Process more videos**: Run the system on your entire dataset

For more information, see:
- `docs/GETTING_STARTED.md`
- `docs/ARCHITECTURE.md`
- `models/README.md`